# Thực nghiệm trên dữ liệu mới: ARSK trên Wine

Thiết kế thực nghiệm trong notebook này:
- Chạy đầy đủ 4 cấu hình ngưỡng theo phong cách Simulation 1:
  - soft-soft
  - soft-SCAD
  - SCAD-soft
  - SCAD-SCAD
- Lặp lại trên 10 seed khác nhau.
- Báo cáo mean +/- std cho CER, lambda được chọn, số outlier dự đoán và độ thưa (số trọng số bằng 0).

## Cell 1: Import thư viện và module dự án

In [1]:
import os
import sys
import time
import numpy as np
import pandas as pd

from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

path_to_src = "../src"
abs_src = os.path.abspath(path_to_src)
if abs_src not in sys.path:
    sys.path.append(abs_src)

from model import arsk, select_lambda
from metrics import compute_cer

## Cấu hình thực nghiệm

Cell này định nghĩa 4 tổ hợp ngưỡng và số lần lặp lại để đảm bảo kết quả có thể so sánh và tái lập.

In [2]:
# Configuration
THRESHOLD_CONFIGS = {
    "soft-soft": ("soft", "soft"),
    "soft-SCAD": ("soft", "scad"),
    "SCAD-soft": ("scad", "soft"),
    "SCAD-SCAD": ("scad", "scad"),
}

N_RUNS = 10
SEEDS = list(range(N_RUNS))

print("Threshold configs:", THRESHOLD_CONFIGS)
print("Seeds:", SEEDS)

Threshold configs: {'soft-soft': ('soft', 'soft'), 'soft-SCAD': ('soft', 'scad'), 'SCAD-soft': ('scad', 'soft'), 'SCAD-SCAD': ('scad', 'scad')}
Seeds: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


## Hàm hỗ trợ và đánh giá từng lần chạy

In [3]:
def calculate_cer_baseline(y_true, y_pred, n_clusters):
    """
    Baseline CER computed with the same metric function as ARSK.
    KMeans has no explicit outlier component, so pass zero errors and no true outliers.
    """
    errors = np.zeros((len(y_true), 1), dtype=float)
    true_outliers = np.zeros(len(y_true), dtype=bool)

    return compute_cer(
        y_true=y_true,
        labels_pred=y_pred,
        errors=errors,
        true_outliers=true_outliers,
        n_clusters=n_clusters,
    )


def prepare_wine_data():
    wine = load_wine()
    X = wine.data
    y = wine.target
    feature_names = wine.feature_names

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    return X_scaled, y, feature_names


def run_single_setting(X_scaled, y_true, seed, thE, thW):
    """
    Run ARSK for one seed and one threshold setting.
    Lambda selection uses robust selector from original paper/code (`select_lambda`).
    """
    k = len(np.unique(y_true))

    # Baseline KMeans for reference
    kmeans = KMeans(n_clusters=k, random_state=seed, n_init=10)
    kmeans.fit(X_scaled)
    cer_kmeans = calculate_cer_baseline(y_true, kmeans.labels_, k)

    # ARSK with robust lambda selector (paper-consistent)
    t0 = time.time()
    lam1_opt, lam2_opt = select_lambda(X_scaled, thresh_E=thE, thresh_w=thW)

    res = arsk(
        X=X_scaled,
        n_clusters=k,
        lambda1=lam1_opt,
        lambda2=lam2_opt,
        thresh_E=thE,
        thresh_w=thW,
        random_state=seed,
    )
    elapsed = time.time() - t0

    labels = res["labels"]
    errors = res["errors"]
    weights = res["weights"]

    # No ground-truth outlier labels on Wine, use all-False mask for CER adapter
    true_outliers = np.zeros(X_scaled.shape[0], dtype=bool)

    cer_arsk = compute_cer(
        y_true=y_true,
        labels_pred=labels,
        errors=errors,
        true_outliers=true_outliers,
        n_clusters=k,
    )

    n_pred_outliers = int(np.sum(np.linalg.norm(errors, axis=1) > 1e-6))
    n_zero_weights = int(np.sum(np.isclose(weights, 0, atol=1e-5)))

    return {
        "cer_kmeans": float(cer_kmeans),
        "cer_arsk": float(cer_arsk),
        "lambda1": float(lam1_opt),
        "lambda2": float(lam2_opt),
        "n_pred_outliers": n_pred_outliers,
        "n_zero_weights": n_zero_weights,
        "time_sec": float(elapsed),
        "weights": weights.copy(),
    }

## Vòng lặp thực nghiệm tổng

Cell này định nghĩa hàm chạy toàn bộ thí nghiệm trên:
- 10 seed
- 4 cấu hình ngưỡng

In [4]:
def run_wine_experiment(n_runs=10):
    X_scaled, y_true, feature_names = prepare_wine_data()

    print("=== IDEA D: ARSK on Wine (new dataset) ===")
    print(f"Data shape: n={X_scaled.shape[0]}, p={X_scaled.shape[1]}")
    print(f"Clusters: {len(np.unique(y_true))}")
    print(f"Runs: {n_runs}")

    all_rows = []

    for run in range(n_runs):
        seed = run
        print(f"\\n--- Run {run + 1}/{n_runs} | seed={seed} ---")

        for name, (thE, thW) in THRESHOLD_CONFIGS.items():
            out = run_single_setting(X_scaled, y_true, seed, thE, thW)

            row = {
                "run": run,
                "seed": seed,
                "config": name,
                "thresh_E": thE,
                "thresh_w": thW,
                "cer_kmeans": out["cer_kmeans"],
                "cer_arsk": out["cer_arsk"],
                "delta_cer_vs_kmeans": out["cer_arsk"] - out["cer_kmeans"],
                "lambda1": out["lambda1"],
                "lambda2": out["lambda2"],
                "n_pred_outliers": out["n_pred_outliers"],
                "n_zero_weights": out["n_zero_weights"],
                "time_sec": out["time_sec"],
                "weights": out["weights"],
            }
            all_rows.append(row)

            print(
                f"{name}: CER_ARSK={out['cer_arsk']:.4f}, CER_KMeans={out['cer_kmeans']:.4f}, "
                f"lam=({out['lambda1']:.3f}, {out['lambda2']:.3f}), "
                f"zero_w={out['n_zero_weights']}, pred_out={out['n_pred_outliers']}, t={out['time_sec']:.2f}s"
            )

    df = pd.DataFrame(all_rows)
    return df, feature_names

## Chạy thí nghiệm chính

Cell này thực sự thực thi workload đầy đủ (4 ngưỡng x 10 lần chạy).

In [5]:
# Run full experiment: 4 thresholds x 10 runs
results_df, feature_names = run_wine_experiment(n_runs=N_RUNS)
results_df.head()

=== IDEA D: ARSK on Wine (new dataset) ===
Data shape: n=178, p=13
Clusters: 3
Runs: 10
\n--- Run 1/10 | seed=0 ---
soft-soft: CER_ARSK=0.5556, CER_KMeans=0.0457, lam=(0.500, 0.500), zero_w=0, pred_out=150, t=31.96s
soft-SCAD: CER_ARSK=0.5556, CER_KMeans=0.0457, lam=(0.500, 3.875), zero_w=0, pred_out=150, t=31.26s
SCAD-soft: CER_ARSK=0.5556, CER_KMeans=0.0457, lam=(0.500, 0.500), zero_w=0, pred_out=150, t=32.05s
SCAD-SCAD: CER_ARSK=0.5556, CER_KMeans=0.0457, lam=(0.500, 3.875), zero_w=0, pred_out=150, t=31.19s
\n--- Run 2/10 | seed=1 ---
soft-soft: CER_ARSK=0.4274, CER_KMeans=0.0457, lam=(0.500, 0.500), zero_w=0, pred_out=118, t=31.69s
soft-SCAD: CER_ARSK=0.4274, CER_KMeans=0.0457, lam=(0.500, 3.875), zero_w=0, pred_out=118, t=31.10s
SCAD-soft: CER_ARSK=0.4368, CER_KMeans=0.0457, lam=(0.500, 0.500), zero_w=0, pred_out=120, t=31.86s
SCAD-SCAD: CER_ARSK=0.4368, CER_KMeans=0.0457, lam=(0.500, 3.875), zero_w=0, pred_out=120, t=31.24s
\n--- Run 3/10 | seed=2 ---
soft-soft: CER_ARSK=0.5556, 

,run,seed,config,thresh_E,thresh_w,cer_kmeans,cer_arsk,delta_cer_vs_kmeans,lambda1,lambda2,n_pred_outliers,n_zero_weights,time_sec,weights
0,0,0,soft-soft,soft,soft,0.045706,0.555577,0.509871,0.5,0.500,150,0,31.963482,"[0.32807621923748226, 0.1865035746851113, 0.07..."
1,0,0,soft-SCAD,soft,scad,0.045706,0.555577,0.509871,0.5,3.875,150,0,31.258049,"[0.3279586447693935, 0.18710252781147413, 0.07..."
2,0,0,SCAD-soft,scad,soft,0.045706,0.555577,0.509871,0.5,0.500,150,0,32.049753,"[0.33208238840394666, 0.18788654082480338, 0.0..."
3,0,0,SCAD-SCAD,scad,scad,0.045706,0.555577,0.509871,0.5,3.875,150,0,31.191468,"[0.33194527013944564, 0.1884756033445016, 0.07..."
4,1,1,soft-soft,soft,soft,0.045706,0.427411,0.381705,0.5,0.500,118,0,31.685898,"[0.3314362801948519, 0.18493312908791512, 0.07..."


## Tổng hợp bảng kết quả

In [6]:
# Summary table: mean ± std by threshold config
summary_df = (
    results_df
    .groupby(["config", "thresh_E", "thresh_w"], as_index=False)
    .agg(
        cer_kmeans_mean=("cer_kmeans", "mean"),
        cer_kmeans_std=("cer_kmeans", "std"),
        cer_arsk_mean=("cer_arsk", "mean"),
        cer_arsk_std=("cer_arsk", "std"),
        delta_cer_mean=("delta_cer_vs_kmeans", "mean"),
        delta_cer_std=("delta_cer_vs_kmeans", "std"),
        lambda1_mean=("lambda1", "mean"),
        lambda1_std=("lambda1", "std"),
        lambda2_mean=("lambda2", "mean"),
        lambda2_std=("lambda2", "std"),
        pred_outliers_mean=("n_pred_outliers", "mean"),
        pred_outliers_std=("n_pred_outliers", "std"),
        zero_weights_mean=("n_zero_weights", "mean"),
        zero_weights_std=("n_zero_weights", "std"),
        time_mean_sec=("time_sec", "mean"),
        time_std_sec=("time_sec", "std"),
    )
)

summary_df = summary_df.sort_values("config").reset_index(drop=True)
summary_df

,config,thresh_E,thresh_w,cer_kmeans_mean,cer_kmeans_std,cer_arsk_mean,cer_arsk_std,delta_cer_mean,delta_cer_std,lambda1_mean,lambda1_std,lambda2_mean,lambda2_std,pred_outliers_mean,pred_outliers_std,zero_weights_mean,zero_weights_std,time_mean_sec,time_std_sec
0,SCAD-SCAD,scad,scad,0.044931,0.002449,0.473396,0.056787,0.428464,0.056284,0.5,0.0,3.875,0.0,129.2,14.366628,0.0,0.0,31.258864,0.206564
1,SCAD-soft,scad,soft,0.044931,0.002449,0.473396,0.056787,0.428464,0.056284,0.5,0.0,0.500,0.0,129.2,14.366628,0.0,0.0,31.909541,0.154602
2,soft-SCAD,soft,scad,0.044931,0.002449,0.467739,0.060729,0.422808,0.060339,0.5,0.0,3.875,0.0,128.0,15.202339,0.0,0.0,31.102839,0.094681
3,soft-soft,soft,soft,0.044931,0.002449,0.467225,0.061041,0.422294,0.060587,0.5,0.0,0.500,0.0,127.9,15.263974,0.0,0.0,31.739815,0.211244


Bảng `summary_df` tổng hợp trung bình và độ lệch chuẩn qua 10 lần chạy cho từng cấu hình.

- `cer_kmeans_mean`, `cer_arsk_mean`: CER trung bình của baseline và ARSK.
- `delta_cer_mean = cer_arsk - cer_kmeans`:
  - `< 0`: ARSK tốt hơn KMeans.
  - `> 0`: ARSK kém hơn KMeans.
- `lambda1_mean`, `lambda2_mean`: mức regularization được chọn bởi robust selector.
- `pred_outliers_mean`: số điểm ARSK đánh dấu outlier (theo norm của `errors`).
- `zero_weights_mean`: số biến bị triệt tiêu (mức độ sparsity).
- `time_mean_sec`: thời gian trung bình mỗi run.

In [7]:
# Print concise report in Simulation-1 style
for _, r in summary_df.iterrows():
    print("=" * 60)
    print(f"{r['config']} ({r['thresh_E']}-{r['thresh_w']})")
    print(f"CER KMeans: {r['cer_kmeans_mean']:.4f} (+/- {r['cer_kmeans_std']:.4f})")
    print(f"CER ARSK  : {r['cer_arsk_mean']:.4f} (+/- {r['cer_arsk_std']:.4f})")
    print(f"Delta CER (ARSK - KMeans): {r['delta_cer_mean']:.4f} (+/- {r['delta_cer_std']:.4f})")
    print(f"lambda1: {r['lambda1_mean']:.3f} (+/- {r['lambda1_std']:.3f})")
    print(f"lambda2: {r['lambda2_mean']:.3f} (+/- {r['lambda2_std']:.3f})")
    print(f"Pred outliers: {r['pred_outliers_mean']:.2f} (+/- {r['pred_outliers_std']:.2f})")
    print(f"Zero weights : {r['zero_weights_mean']:.2f} (+/- {r['zero_weights_std']:.2f})")
    print(f"Time per run : {r['time_mean_sec']:.2f}s (+/- {r['time_std_sec']:.2f}s)")

SCAD-SCAD (scad-scad)
CER KMeans: 0.0449 (+/- 0.0024)
CER ARSK  : 0.4734 (+/- 0.0568)
Delta CER (ARSK - KMeans): 0.4285 (+/- 0.0563)
lambda1: 0.500 (+/- 0.000)
lambda2: 3.875 (+/- 0.000)
Pred outliers: 129.20 (+/- 14.37)
Zero weights : 0.00 (+/- 0.00)
Time per run : 31.26s (+/- 0.21s)
SCAD-soft (scad-soft)
CER KMeans: 0.0449 (+/- 0.0024)
CER ARSK  : 0.4734 (+/- 0.0568)
Delta CER (ARSK - KMeans): 0.4285 (+/- 0.0563)
lambda1: 0.500 (+/- 0.000)
lambda2: 0.500 (+/- 0.000)
Pred outliers: 129.20 (+/- 14.37)
Zero weights : 0.00 (+/- 0.00)
Time per run : 31.91s (+/- 0.15s)
soft-SCAD (soft-scad)
CER KMeans: 0.0449 (+/- 0.0024)
CER ARSK  : 0.4677 (+/- 0.0607)
Delta CER (ARSK - KMeans): 0.4228 (+/- 0.0603)
lambda1: 0.500 (+/- 0.000)
lambda2: 3.875 (+/- 0.000)
Pred outliers: 128.00 (+/- 15.20)
Zero weights : 0.00 (+/- 0.00)
Time per run : 31.10s (+/- 0.09s)
soft-soft (soft-soft)
CER KMeans: 0.0449 (+/- 0.0024)
CER ARSK  : 0.4672 (+/- 0.0610)
Delta CER (ARSK - KMeans): 0.4223 (+/- 0.0606)
lambda1: 

## Phân tích độ thưa theo từng đặc trưng

In [ ]:
weight_rows = []
for _, row in results_df.iterrows():
    w = row["weights"]
    for j, wj in enumerate(w):
        weight_rows.append({
            "run": row["run"],
            "config": row["config"],
            "feature": feature_names[j],
            "weight": float(wj),
            "is_zero": bool(np.isclose(wj, 0, atol=1e-5)),
        })

weights_long_df = pd.DataFrame(weight_rows)

zero_feature_rate_df = (
    weights_long_df
    .groupby(["config", "feature"], as_index=False)
    .agg(zero_rate=("is_zero", "mean"), mean_weight=("weight", "mean"))
    .sort_values(["config", "zero_rate", "mean_weight"], ascending=[True, False, True])
)

zero_feature_rate_df.head(30)

,config,feature,zero_rate,mean_weight
2,SCAD-SCAD,ash,0.0,0.071127
6,SCAD-SCAD,magnesium,0.0,0.114619
0,SCAD-SCAD,alcalinity_of_ash,0.0,0.138186
8,SCAD-SCAD,nonflavanoid_phenols,0.0,0.152142
10,SCAD-SCAD,proanthocyanins,0.0,0.161162
7,SCAD-SCAD,malic_acid,0.0,0.187079
5,SCAD-SCAD,hue,0.0,0.293947
12,SCAD-SCAD,total_phenols,0.0,0.306197
3,SCAD-SCAD,color_intensity,0.0,0.311766
1,SCAD-SCAD,alcohol,0.0,0.334444


## Toàn bộ kết quả

In [9]:
print("=" * 80)
print("RAW RESULTS (all runs)")
print("=" * 80)
print(results_df.drop(columns=["weights"]).to_string(index=False))

print("\n" + "=" * 80)
print("SUMMARY (mean ± std by threshold config)")
print("=" * 80)
print(summary_df.to_string(index=False))

print("\n" + "=" * 80)
print("ZERO-FEATURE RATE (top 30)")
print("=" * 80)
print(zero_feature_rate_df.head(30).to_string(index=False))

print("\nCompleted: printed all outputs, no files were saved.")

RAW RESULTS (all runs)
 run  seed    config thresh_E thresh_w  cer_kmeans  cer_arsk  delta_cer_vs_kmeans  lambda1  lambda2  n_pred_outliers  n_zero_weights  time_sec
   0     0 soft-soft     soft     soft    0.045706  0.555577             0.509871      0.5    0.500              150               0 31.963482
   0     0 soft-SCAD     soft     scad    0.045706  0.555577             0.509871      0.5    3.875              150               0 31.258049
   0     0 SCAD-soft     scad     soft    0.045706  0.555577             0.509871      0.5    0.500              150               0 32.049753
   0     0 SCAD-SCAD     scad     scad    0.045706  0.555577             0.509871      0.5    3.875              150               0 31.191468
   1     1 soft-soft     soft     soft    0.045706  0.427411             0.381705      0.5    0.500              118               0 31.685898
   1     1 soft-SCAD     soft     scad    0.045706  0.427411             0.381705      0.5    3.875              118   